---
title: Values and Variables
abstract: |
    In this notebook, readers will see how different expressions in a computer program get evaluated to different types of values. With variables, programmers can assign a meaningful name, known as an identifier, to a value of various types without having to worry about where it should be stored and transferred in the physical storage unit for computations. At this point, readers may regard identifiers simply as variables that can take different values, although this model will be refined later with the concepts of aliasing and mutations.
skip_execution: true
---

In [ ]:
import sys  # to access system-specific parameters
from dis import dis  # for disassembly of bytecode
from ipywidgets import interact  # for interactive user interface controls

## Values

Programming is the art of instructing a computer to perform tasks by manipulating data, which is represented as values. In mathematical terms, a computer (programming language) is essentially a set of rules for how values are stored and manipulated. In other words, values are the building blocks of computation, and they represent a specific piece of data that can be manipulated in various ways.

:::::{important} Values in Turing Machine
:class: dropdown

For the [Turing Machine](https://en.wikipedia.org/wiki/Turing_machine), a program that specifies which values to compute is itself a value. The program and the data it manipulates are mingled together into a tape of binary sequence:

::::{card}
:header: How Do Turing Machines Work?
:footer: [open in new tab](https://www.youtube.com/embed/-ZS_zFg4w5k?si=ni226Dui_ikRBoSy)

:::{iframe} https://www.youtube.com/embed/-ZS_zFg4w5k?si=ni226Dui_ikRBoSy
:width: 100%
:::

::::

:::::

### Integers

While a machine language only uses binary numbers as values, a high-level programming language provides more flexible and expressive ways to represent and manipulate values.

In Python, for instance, we can enter an [integer](https://docs.python.org/3/reference/lexical_analysis.html#integer-literals) in different number systems as follows.

In [ ]:
# in decimal (base 10)
15

In [ ]:
# in 'b'inary (base 2)
0b1111

In [ ]:
# in 'o'ctadecimal (base 8)
0o17

In [ ]:
# in he'x'adecimal (base 16)
0xF

All the above expressions are *integer literals*, namely, integers written out literally. They have the same numerical value, which gets printed in decimal by default.

::::{caution}

Later in the course, you will learn that different objects in Python can share
the same value. In fact, value equality is a comparison operation that can be
defined by the programmer.

::::

In [ ]:
%%hermes --no-context
Explain briefly how to redefine the equality operator in Python so that 
0 is equal to 1?

::::{seealso}
:class: dropdown

In Python, you **can't** modify the `__eq__` method on the built-in `int` type (it's implemented in C), so you can't make the literal `0 == 1` evaluate to `True`.

However, you **can** create a custom integer subclass where it happens:

```python
class WeirdInt(int):
    def __eq__(self, other):
        if isinstance(other, WeirdInt):
            return True  # everything equals everything
        return super().__eq__(other)

a = WeirdInt(0)
b = WeirdInt(1)
print(a == b)  # True
```

Key limitation: `a == 0` is `True`, but `a == 1` returns `False` (because `a.__eq__(1)` falls through to `super().__eq__`), i.e., the built-in integer comparison.
::::

There are also expressions with integer values but are not integer literals:

In [ ]:
4 + 5 + 6

In [ ]:
pow(2, 4) - 1

::::{exercise} bignum
:label: ex:big-integer

Enter an expression that evaluates to an integer value as big as possible.

:::{hint}
:class: dropdown

Try `sys.get_int_max_str_digits()` for limits, or simply compute `10 ** 4000` — Python integers are unbounded.

:::

::::

In [ ]:
# YOUR CODE HERE
raise NotImplementedError
# SPOILER: Printing an integer actually involves converting it to a string!

::::{seealso} How to represent all integers?
:class: dropdown

Wouldn't it be nice if there is no limit on how many integers a computer can
represent? Since $n$ bits can represent at most $2^n$ integers, do we need
infinite bits to represent an unbounded number of integers? That would take
forever to input an integer into the computer, let alone storing it in finite
memory! How about using a variable-length code? See the [Kraft-McMillan
inequality](https://en.wikipedia.org/wiki/Kraft%E2%80%93McMillan_inequality).
An example of variable-length code is [UTF-8 encoding for
Unicode](https://en.wikipedia.org/wiki/UTF-8#Description).

::::

In [ ]:
%%hermes --no-context
Use a simple analogy to explain in a short paragraph how can a computer 
represent all integers using variable-length code?

::::{seealso}
:class: dropdown

Think of it like a game of "hot or cold" where you ask yes/no questions to guess a number. To represent small numbers, you need just a few bits—like asking "is it zero?" and stopping immediately if the answer is yes. For larger numbers, you ask more questions: "is the high bit set?", then "is the next bit set?", and so on. The trick is using a continuation bit—a prefix that says "keep reading" or "this is the end." UTF-8 does something similar for text: single-byte characters (common ASCII) get short codes, while rarer characters use multi-byte sequences where each byte except the last has its high bit set as a signal to continue. This way, you can represent any integer: small ones use few bits, and the code simply grows longer as needed.
::::

### Strings

A string value is a sequence of characters that can be written literally using quotes:

In [ ]:
# single quote
print("\U0001f600:\n\tI'm a string.")

In [ ]:
# double quote
print("\N{grinning face}:\n\tI'm a string.")

In [ ]:
# triple double quote
print(
    """😀:
	I'm a string."""
)

Note that all the literals represent the same value:

::::{grid} 1 1 2 2

:::{card}
:header: Escape sequence

`\U0001f600` and `\N{grinning face}` are *escape sequences* representing the same grinning face emoji 😀 where

- `0001f600` is the unicode in hexadecimal,
- `grinning face` is the [name][CLDR], and
- `\` is called the *escape symbol*.

[CLDR]: https://unicode.org/emoji/charts/full-emoji-list.html

:::

:::{card}
:header: Control code

`\n` and `\t` are *control code* that does not represent any symbol.
- `\n` creates a new line when printing the string.  
- `\t` creates a tab to indent the line.

:::

::::

::::{note} Benefits of allowing single, double, and triple quotes for string literals
:class: dropdown

- By using double quotes, we don't need to escape the single quote in strings such as "I'm".
- Triple quotes enable a multi-line string literal to include the newline character directly, resulting in a more readable representation of the literal.

::::

Why a string value is often quoted in a computer program? Let's ask AI:

In [ ]:
%%hermes --no-context
Say your name.

In [ ]:
%%hermes --no-context
Say "your name".

An example is worth a thousand words, but exceptions often exist:

In [ ]:
%%hermes --no-context
Are there programming languages that do not quote its string? Why?

::::{seealso}
:class: dropdown

Several languages don't require quotes for strings, and the reasons reveal interesting design trade-offs:

**Text-as-default languages** — In LaTeX, Markdown, HTML, the default content *is* text. Special syntax (like `$...$` for math in LaTeX) marks the non-text parts. This makes sense because these languages are primarily about documents, where code is the exception.

**Shell scripting** — In bash, unquoted `hello` is treated as a string (a word that may undergo expansion/splitting). It works because the shell's grammar distinguishes command positions from data positions by context.

**Prolog** — Unquoted `hello` is an *atom* (a symbolic constant), while `"hello"` is actually a list of character codes. The language distinguishes symbolic identifiers from character sequences.

**Why Python quotes strings** — The core ambiguity: in `hello`, is it a variable name or the string `"hello"`? Without quotes, the parser can't tell. Quoting is the simplest way to disambiguate.

The notebook's point connects here: quoting strings in programs separates *data* (the string value) from *code* (identifiers, keywords, operators). In document languages, that distinction is flipped — text is data by default, and markup is the annotation.

::::

The following is yet another way to print the same string:

In [ ]:
print("\N{grinning face}:", "\tI'm a string.", sep="\n")

It is an elegant [one-liner](https://en.wikipedia.org/wiki/One-liner_program) where

- `sep="\n"` is a *keyword argument* that specifies the separator of the list of strings.
- The default separator is a single space character, i.e., `sep=" "`.

In a notebook, we can get the *docstring* (document string) of a function conveniently using the symbol `?` such as `?print` or

In [ ]:
print?

We can also use the contextual help by placing the cursor over a function name and 
- click the menu item `Help`$\to$`Show Contextual Help` or
- press the short-cut key <kbd>Shift + Tab</kbd>.

::::{exercise} multi-line string
:label: multi-line

Print an intriguing multi-line string below. Ensure that your string is complex enough to confound a machine, yet comprehensible to a human.

:::{seealso}
:class: dropdown

- `art`: <https://www.ascii-art.site>
- `bashplotlib`: <https://github.com/glamp/bashplotlib/blob/master/README.md>
- Star Wars via Telnet: <http://asciimation.co.nz/>
  You can also try running `ssh starwarstel.net` in a terminal.

:::

::::

In [ ]:
# YOUR CODE HERE
raise NotImplementedError

Check if AI can understand your message, such as the ones below:

In [ ]:
%%hermes --no-context
Explain what you see in the following:
 (ง •̀_•́)ง 
 ╰(●’◡’●)╮ 
 (..•˘_˘•..)
 (づ￣ 3￣)づ

::::{seealso}
:class: dropdown

These are **kaomoji** (Japanese emoticons) — visual expressions built entirely from Unicode characters arranged to form faces and gestures. They're a perfect example of string values as sequences of characters:

1. **(ง •̀_•́)ง** — "Flexing muscles" / "Ready to fight" — raised fists
2. **╰(●'◡'●)╮** — "Shrug" / happy open-arms gesture
3. **(..•˘_˘•..)** — Sleepy / tired face
4. **(づ￣ 3￣)づ** — Hugging gesture

Each one is a string value composed of Unicode characters from different blocks — parentheses, Latin letters, special symbols, and decorative characters. They demonstrate that strings aren't just letters and numbers; they're arbitrary sequences of code points that, when rendered by a font, form visual meaning.
::::

### User Input

Instead of entering a value in a program, a programmer can get user input values at *runtime*, i.e., when a program executes:

In [ ]:
print("Your name is", input("Please input your name: ") + ".")

- The `input` method prints its argument, if any, as a [prompt](https://en.wikipedia.org/wiki/Command-line_interface#Command_prompt).  
- The method takes user's input and *returns it as a string*.
- There is no need to delimit the input string by quotation marks. Simply press `enter` after typing a string.

::::{exercise}
:label: ex:print-returns-none

Explain whether the following code prints `'My name is Python'`. 

:::{hint}
:class: dropdown

Does `print` return a value? 
:::
::::

In [ ]:
print("My name is", print("Python"))

YOUR ANSWER HERE

In [ ]:
%%hermes --no-context
Explain whether the Python print function returns the value it prints.

::::{seealso}
:class: dropdown

No — `print()` returns `None`. It's a **procedure** (side-effect), not a function that computes a value.
::::

## Variables

A complicated computation often needs to be broken down into many basic computations, with intermediate values stored and transferred to different memory locations. Keeping track of where a value is written, and allocating free memory locations to write to, are not only burdensome but also error-prone.

This is where the concept of variables comes in—they serve as a logical (as opposed to physical) unit of storage that abstracts away the complexities of memory management.

### Assignment

To define a variable, we can use the assignment operator `=` as follows:

In [ ]:
x = 15

What does the above code mean?

1. $x$ is equal to $15$?
2. $x$ is defined to be $15$?

Let's discover the truth by executing the following assignments step-by-step using [`%%pytutor` magic](https://github.com/dive4dec/jupyter-python-tutor):

In [ ]:
%%pytutor
x = 15
x = x + 1

In the second assignment, does it mean:

1. $x$ is equal to $x$+1?
2. $x$ is defined to be $x$+1?

If we say yes to the above questions, the value of $x$ should be $\pm\infty$.
(Why?)

In [ ]:
%%hermes --no-context
What is the solution to $x+1=x$ for $x$ in the extended real line?

::::{seealso}
:class: dropdown

In the extended real line $\mathbb{R} \cup \{-\infty, +\infty\}$, both **$+\infty$** and **$-\infty$** are solutions. For any finite real number $x$, subtracting $x$ from both sides gives $1 = 0$ (impossible). But by the arithmetic conventions of the extended reals: $(+\infty) + 1 = +\infty$ and $(-\infty) + 1 = -\infty$.

::::

To see how the Python interpreter carries out the assignment operation, the following code compiles the Python code to bytecode (similar to machine code) and displays it in assembly language.[^dis]

[^dis]: The conversion from machine code to assembly code is called disassembly, hence the name [`dis`](https://docs.python.org/3/library/dis.html). BTW, why an interpreted language like Python has a compiler? The interpreter actually compiles the source code to a more compact form called the bytecode that allows the interpreter to run it faster. Indeed, even though Python was originally intended to be an interpreted language, Python 3.13 supports [JIT compilation](https://peps.python.org/pep-0744/) that will compile source code **J**ust **I**n **T**ime to machine code for the first run... Why? See [this post](https://tonybaloney.github.io/posts/python-gets-a-jit.html).

In [ ]:
dis(compile('x = x + 1', '_', 'exec'))

It is possible to assign different values to multiple variables in one line using the so-called *tuple assignment* syntax:

In [ ]:
x, y, z = "15", "30", 15
x, y, z

One can also assign the same value to different variables in one line using a
*chained assignment*:

In [ ]:
x = y = z = 0
x, y, z

Once defined, a variable can be deleted using the `del` keyword. Accessing a variable that has not been assigned any value raises an error.

In [ ]:
x = y = 1+1j
del x
x

::::{caution}
:class: dropdown

You will learn later in the course that deleting a variable does not necessarily delete its value. In the above example, the value `1+1j` continues to exist after `x` is deleted.

::::

::::{seealso} Are there constants in Python?
:class: dropdown

As the concepts of variables and values in programming are analogous to those
in mathematics. Is there a programming counterpart for mathematical constants
such as $\pi$? In Python:

- There is a list of
[constants](https://docs.python.org/3/library/constants.html).
- There are [immutable
objects](https://docs.python.org/3/glossary.html#term-immutable).
- There is a [math module defining mathematical constants like
`pi`](https://docs.python.org/3/library/math.html#math.pi).

::::

In [ ]:
%%hermes --no-context
How would you define constants in Python? Are literals also constants?

::::{seealso}
:class: dropdown

Constants in Python:

- Python has **no built-in `const` keyword** — there's no language-level enforcement
- **Convention**: `ALL_CAPS` names (e.g., `MAX_SIZE = 100`) signal intent
- **Type-checker support**: `typing.Final` prevents reassignment at static analysis time

**Are literals constants?** Not in the same sense. A literal (`42`, `"hello"`) is a value written directly in code. A constant is a *named variable* intended not to change. Literals are inherently fixed (you can't modify `42`), but they aren't "constants" — they're *values*. Constants are names bound to values that you promise not to rebind.
::::

### Identifiers

One reason why Python is expressive is that it affords programmers a significant amount of flexibility when it comes to choosing variable names. For instance, *identifiers*, such as variable names, are case-sensitive and of unlimited length, unlike older languages such as Pascal and Fortran. This flexibility also makes the program more readable. For instance, consider the following program:

In [ ]:
def name():
    return first.name + last.name

first.name = "John"
last.name = "Smith"
print(name())

Unfortunately, the program fails in the middle, why? Let's take a closer look at the operations involved:

In [ ]:
dis(compile("first.name + last.name", "_", "eval"))

How about the following fix?

In [ ]:
dis(compile("first-name + last-name", "_", "eval"))

Obviously, not all names are valid identifiers as some names may be
misinterpreted by the Python interpreter. Instead of fixing the names by
trial-and-error, let's try to learn the exact [syntax for
identifiers](https://docs.python.org/3/reference/lexical_analysis.html#identifiers)

```ebnf
identifier   ::=  xid_start xid_continue*
...
```

which is specified in a notation called the [Extended Backus-Naur Form
(EBNF)](https://en.wikipedia.org/wiki/Extended_Backus%E2%80%93Naur_form). Some
identifiers called
[*keywords*](https://docs.python.org/3/reference/lexical_analysis.html#keywords)
are reserved and therefore not available for programmers to use as variable
names. There are also [*soft
keywords*](https://docs.python.org/3/reference/lexical_analysis.html#soft-keywords)
that are reserved under specific contexts.

In [ ]:
%%hermes --no-context
Explain in one paragraph the notation of the following in pydoc:
identifier ::= xid_start xid_continue*

::::{seealso}
:class: dropdown

This is **ABNF (Augmented Backus-Naur Form)** notation. The `::=` means "is defined as." It says an identifier must start with one character from `xid_start` (letters, `_`, etc.) followed by zero or more characters (`*`) from `xid_continue` (letters, digits, `_`). So `x` is valid (one start char, zero continues), but `3x` is not (digits aren't in `xid_start`).

::::

::::{tip}

Consider the following simpler rules of thumb:

1. Start a variable name with a letter or `_` (an underscore) followed by
letters, digits, or `_`, but
2. do not use any of the following reserved words:

    ```python
    False      await      else       import     pass
    None       break      except     in         raise
    True       class      finally    is         return
    and        continue   for        lambda     try
    as         def        from       nonlocal   while
    assert     del        global     not        with
    async      elif       if         or         yield
    ```

That should work most of the time except for a few specific
cases.[^soft-keywords]

::::

[^soft-keywords]: `match` and `case` are recognized as keywords in the context
of a [`match`
statement](https://docs.python.org/3/tutorial/controlflow.html#tut-match),
where `_` is treated as a wildcard. This is regardless of whether `match`,
`case`, or `_` are used as variables outside this context.

In [ ]:
%%hermes --no-context
What is the motivation for soft keywords in Python? Why not make them hard?

::::{seealso}
:class: dropdown

**Backward compatibility.** Making a word a hard keyword breaks all existing code that uses it as an identifier (variable names, function names, etc.). Soft keywords are only special in their specific context (`match`/`case` only at the start of a statement, `type` only in type alias positions, `_` only in pattern matching), so existing code like `match = "winner"` or `case = 1` continues to work unchanged.

::::

::::{exercise}
:label: ex:identifiers-syntax

Evaluate the following cell and check if any of the rules above is violated.

:::{hint}
:class: dropdown

- `del` is a keyword and `Del` is not because identifiers are case sensitive.
- Function names such as `print`, `input`, `type`, etc., are not keywords and can be reassigned.  
  This can be useful if you want to modify the default implementations without changing their source code.
:::

::::

In [ ]:
@interact
def identifier_syntax(
    assignment=[
        "a-number = 15",
        "a_number = 15",
        "15 = 15",
        "_15 = 15",
        "del = 15",
        "Del = 15",
        "type = print",
        "print = type",
        "input = print",
    ]
):
    exec(assignment)
    print("Ok.")

YOUR ANSWER HERE

::::{seealso} Python Enhancement Proposals
:class: dropdown

To help make the code more readable, programmers follow additional style guides such as [Python Enhancement Proposals (PEP) 8](https://www.python.org/dev/peps/pep-0008/#function-and-variable-names):
- Function names should be lowercase, with words separated by underscores as necessary to improve readability.  
- Variable names follow the same convention as function names.

::::

::::{exercise} agentic naming
:label: ex:identifier-agentic

Ask an AI agent to suggest valid Python identifiers for a given scenario,
then evaluate whether they follow PEP 8 conventions.

::::

YOUR ANSWER HERE

### Type Conversion

The following program tries to compute the sum of two numbers from user inputs:

In [ ]:
num1 = input("Please input an integer: ")
num2 = input("Please input another integer: ")
print(num1, "+", num2, "is equal to", num1 + num2)

::::{exercise}
:label: ex:bug

Unfortunately, there is a [bug](https://en.wikipedia.org/wiki/Software_bug) in the above code. Can you locate the error?

::::

YOUR ANSWER HERE

`input` *returns* user input as a string.  E.g., if the user enters `12`, the input is
- not treated as the integer twelve, but rather
- treated as a string containing two characters, '1' followed by '2'.

To confirm this, we can use `type` to return the data type of an expression.

In [ ]:
num1 = input("Please input an integer: ")
print("Your input is", num1, "with type", type(num1))

::::{exercise}

`type` can apply to any expressions. Try it out below.

::::

In [ ]:
# YOUR CODE HERE
raise NotImplementedError

What happens when we add strings together?

In [ ]:
"4" + "5" + "6"

How to fix the bug then?

We can convert a string to an integer using `int`.

In [ ]:
int("4") + int("5") + int("6")

We can also convert an integer to a string using `str`.

In [ ]:
str(4) + str(5) + str(6)

::::{exercise}
:label: ex:fix

Fix the bug in the following cell.
::::

In [ ]:
num1 = input("Please input an integer: ")
num2 = input("Please input another integer: ")
# print(num1, '+', num2, 'is equal to', num1 + num2)  # fix this line below
# YOUR CODE HERE
raise NotImplementedError

In [ ]:
%%hermes --no-context
What is wrong with this code? Explain the error and fix it.
age = input("Enter your age: ")
print("Next year you will be", age + 1)

::::{seealso}
:class: dropdown


`input()` returns a **string**, so `age + 1` tries to concatenate a string with an integer — which raises a `TypeError`.

**Fix:** convert the input to an integer first:

```python
age = int(input("Enter your age: "))
print("Next year you will be", age + 1)
```

::::

### Error Types

In addition to writing code, a programmer spends significant time in *debugging* code that contains errors. A natural question is:

**Can an error be automatically detected by the computer?**

You have just seen an example of a *logical error*, which is due to an error in the logic. The ability to debug or even detect such an error is, unfortunately, beyond Python interpreter's intelligence.

Let's see if LLM can debug logical error:

In [ ]:
%%hermes --no-context
What is wrong with the following code?
--
num1 = input("Please input a number: ")
num2 = input("Please input another number: ")
print(num1, "+", num2, "is equal to", num1 + num2)

::::{seealso}
:class: dropdown

Same issue — `input()` returns strings, so `num1 + num2` performs **string concatenation**, not numeric addition. If you enter `3` and `4`, it prints `3 + 4 is equal to 34` instead of `7`.

**Fix:** convert to numbers before adding:

```python
num1 = float(input("Please input a number: "))
num2 = float(input("Please input another number: "))
print(num1, "+", num2, "is equal to", num1 + num2)
```

(Using `float` instead of `int` handles decimals as well.)

::::

::::{tip} Can an LLM detect logical errors?
:class: dropdown

The response depends heavily on the context. For example, suppose the above program intended to:

- Add two *binary* numbers from user input.

Typically, numbers refer to real numbers, so LLM likely will get it wrong without knowing the context. Refining questions based on the LLM’s responses can yield more meaningful answers. Essentially, we should [learn to ask questions properly with context to gain knowledge](https://translate.google.com/?hl=en&sl=auto&tl=en&text=%22%E5%AD%B8%E5%95%8F%22%20%3D%3D%20%22%E5%AD%B8%22%20%2B%20%22%E5%95%8F%22%0A%0A&op=translate).

::::

Other kinds of error may be detected automatically by Python. As an example, note that equality `==` is not equal to assignment `=`:

In [ ]:
print("Assignment:")
"學問" = "學" "問"

As another example, juxtaposition is not the same as addition `+`:

In [ ]:
print("Juxtaposition:")
3 == 2 1

Python interpreter detects the bug and raises a *syntax* error.

::::{note} Why Syntax error can be detected automatically?
:class: dropdown

Note that the print statement is not executed. Why?

The Python interpreter can detect a syntax error even before executing the code because the interpreter simply fails to translate the code to lower-level executable code. 

::::

The following code raises a different kind of error.

In [ ]:
print("Add integer to string:")
"4" + "5" + 6  # adding string to integer

::::{note} Why Python throws a TypeError when evaluating `'4' + '5' + 6`?
:class: dropdown

There is no implementation of `+` operation on a value of type `str` and a value of type `int`. 
- Unlike the syntax error, the Python interpreter can only detect a type error at runtime (when executing the code.) 
- Hence, such an error is called a *runtime error*.

::::

Python is a [*strongly-and-dynamically-typed*](https://en.wikipedia.org/wiki/Strong_and_weak_typing) language:

- *Dynamically-typed* languages check data type only at runtime after translating the code to machine code.
- *Strongly-typed*: languages do not force a type conversion to avoid a type error.

To understand what the above actually means, let's consider how other languages work differently.

C/C++, Java, and Rust are *statically-typed* languages that check data type during compilation, so the type error above becomes a compile-time error instead of a runtime error.

In [ ]:
%%bash
g++ -fsyntax-only 456.cpp

In [ ]:
%%bash
javac -d _private/javac-check 456.java

In [ ]:
%%bash
rustc --emit=metadata 456.rs

On the opposite extreme, the web programming language JavaScript does not raise any error at all:

In [ ]:
%%javascript
let x = '4' + '5' + 6;
element.append(x + ' ' + typeof(x));
// no error because 6 is converted to a str implicitly

In [ ]:
%%javascript
let x = '4' + '5' - 6;
element.append(x + ' ' + typeof(x));
// no error because 4 and 5 are converted to numbers implicitly

JavaScript forces a type conversion to make the code run. It is called a *weakly-typed* language because of this flexibility.

(wtfjs)=
::::{seealso} Why not make Python a more flexible weakly-typed language by automatic type conversion?
:class: dropdown

While weakly-typed languages may seem more robust, they can potentially lead to more logical errors. JavaScript, despite its popularity, is known for its tricky behavior, as demonstrated in [wtfjs](https://github.com/denysdovhan/wtfjs). To improve readability and avoid logical errors, it is recommended to use the strongly-typed language TypeScript.[^typescript]

::::

[^typescript]: TypeScript is a superset of JavaScript that adds optional static typing and other features to the language. By enforcing stronger typing, TypeScript can detect potential errors at compile-time and improve the overall reliability of your code. Learn more about TypeScript at [typescriptlang.org](https://www.typescriptlang.org/).

::::{exercise} BANANA program
:label: ex:weakly-typed

Explain why the following JavaScript prints `BANANA`, even though the code is missing the string literal `"N"` or `"n"`.

::::

In [ ]:
%%javascript
element.append(
    ("b" + "a" +
     + "a" + "a").toUpperCase()
);

YOUR ANSWER HERE

Maybe AI knows why?

In [ ]:
%%hermes --no-context
Why does the following JavaScript return BANANA?
element.append(
    ("b" + "a" +
     + "a" + "a").toUpperCase()
);

::::{seealso}
:class: dropdown

It's a classic JavaScript pun! Here's the breakdown:

`"b" + "a"` → `"ba"`

Then `+ + "a"`: the second `+` is the **unary plus operator**, which converts its operand to a number. `+"a"` evaluates to `NaN`.

So `"ba" + NaN` → `"baNaN"` (strings concatenate with `NaN`), and adding the final `"a"` gives `"baNaNa"`.

`.toUpperCase()` turns it into `"BANANA"`. 🍌

::::

In [ ]:
%%hermes --no-context
Why `"4"+"5"+6` works for java while `'4'+'5'+'6'` works for C++? 
Is there a rigorously definition of coersion for weakly-typed language that
is different from the above behavior?

::::{seealso}
:class: dropdown

**Java** — `"4"+"5"+6` works because:
- `+` with at least one `String` operand does **string concatenation**
- Non-strings are implicitly converted via `String.valueOf()` → result `"456"`

**C++** — `'4'+'5'+'6'` works because:
- Single quotes are `char` literals, which are **integral types** (ASCII 52, 53, 54)
- `+` performs numeric addition → 159

**Rigorous coercion definitions?** Yes. The **ECMAScript specification** (JavaScript) defines coercion as precise abstract operations: `ToNumber`, `ToString`, `ToPrimitive`, etc. For example, `+` uses the abstract operation `Add` with explicit rules (if either operand is a string, convert both to strings; otherwise to numbers). So JavaScript's coercion is fully specified—it's just *weak*, not *undefined*.

The key distinction: **weak vs strong** typing is about how *permissive* the rules are, not whether they exist. A weakly-typed language can have a rigorous spec that permits more implicit conversions; a strongly-typed one (like Python) has equally rigorous rules that *reject* implicit coercion between types.

::::